
# TEXT CLASSIFICATION (Phishing Detection) - Proof Of Concept

Flow

```
Text → Tokenizer → Tokens → Model → Logits → Softmax → Label
```

Took 3 minutes on Colab - GPU

In [6]:


# 1. IMPORTS
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer
)
import numpy as np
from datasets import Dataset, DatasetDict

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

train_data = {
    "text": [
        "Your account has been suspended click here to verify",
        "Meeting scheduled at 10 AM tomorrow",
        "Update your bank details immediately",
        "Lunch at 1 PM?",
        "You won a lottery claim now",
        "Project deadline extended to next week",
        "Verify your password to avoid suspension",
        "Let's catch up this weekend"
    ],
    "labels": [1, 0, 1, 0, 1, 0, 1, 0] # 1 -> phishing, 0 -> Mo phishing
}

test_data = {
    "text": [
        "Click here to reset your password",
        "Team outing planned for Friday"
    ],
    "labels": [1, 0]
}

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "test": Dataset.from_dict(test_data)
})

MODEL_NAME = "bert-base-uncased"


In [ ]:
# 2. TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

dataset = dataset.map(preprocess, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

# 3. TRAINING SETUP
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",   # IMPORTANT
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,  # nice for demo
)




# 4. METRICS

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


# 5. TRAINER
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset.get("test"),
    # tokenizer=tokenizer, # not needed
    data_collator=data_collator,
    compute_metrics=compute_metrics # if TASK == "classification" else None
)

# 6. TRAIN
trainer.train()

# 7. SAVE
trainer.save_model("my_model")
tokenizer.save_pretrained("my_model")


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.715735,0.564080,1.000000,1.000000,1.000000,1.000000
2,0.603006,0.531401,1.000000,1.000000,1.000000,1.000000
3,0.556602,0.507534,1.000000,1.000000,1.000000,1.000000
4,0.496325,0.496009,1.000000,1.000000,1.000000,1.000000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my_model/tokenizer_config.json', 'my_model/tokenizer.json')

In [8]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="my_model",
    tokenizer="my_model"
)

test_samples = [
    "Your account is locked, click to verify",
    "Let's have a meeting tomorrow",
    "You have won $5000, claim now",
    "Reminder: project discussion at 5 PM"
]

results = classifier(test_samples)

for text, result in zip(test_samples, results):
    print(f"Text: {text}")
    print(f"Prediction: {result}")
    print("-" * 50)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: Your account is locked, click to verify
Prediction: {'label': 'LABEL_1', 'score': 0.6674430966377258}
--------------------------------------------------
Text: Let's have a meeting tomorrow
Prediction: {'label': 'LABEL_0', 'score': 0.5705237984657288}
--------------------------------------------------
Text: You have won $5000, claim now
Prediction: {'label': 'LABEL_1', 'score': 0.6225012540817261}
--------------------------------------------------
Text: Reminder: project discussion at 5 PM
Prediction: {'label': 'LABEL_0', 'score': 0.556246817111969}
--------------------------------------------------
